### **Mount Drive & Setup**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/mResearch"
%cd $PROJECT_PATH

## **Install Dependencies**

In [ ]:
!pip install -q tensorflow scikit-learn pandas matplotlib tqdm

## **Fix Python Path**

In [ ]:
import sys
import os

sys.path.append(PROJECT_PATH)

print("✅ Project path added")

## **Import Core Modules**

In [ ]:
from src.config import Config
from experiments.run_kfold import main as run_kfold
from experiments.external_validation import main as run_external

## **CREATE CONFIG (SINGLE SOURCE CONTROL)**

In [ ]:
cfg = Config()

# =========================
# EXPERIMENT CONTROL
# =========================
cfg.EXPERIMENT_NAME = "fusion_model_1"
cfg.FUSION_STRATEGY = "model_1"   # or "model_2"

# =========================
# DATA
# =========================
cfg.TRAIN_PATH = os.path.join(PROJECT_PATH, "datasets/training")
cfg.EXTERNAL_PATH  = os.path.join(PROJECT_PATH, "datasets/testing")

# =========================
# TRAINING
# =========================
cfg.BATCH_SIZE = 4
cfg.EPOCHS = 30
cfg.N_SPLITS = 5

# =========================
# METRICS
# =========================
cfg.BOOTSTRAP_SAMPLES = 1000
cfg.CI_ALPHA = 0.95

# =========================
# AUGMENTATION
# =========================
cfg.USE_AUGMENTATION = True

cfg.set_seed()

print("✅ Config ready")

## **RUN K-FOLD TRAINING (MODEL 1)**

In [ ]:
print("🚀 Running K-Fold Training...")

run_kfold(cfg)

## **RUN EXTERNAL VALIDATION (MODEL 1)**

In [ ]:
print("🌍 Running External Validation...")

run_external(cfg)

## **SET-UP FOR MODEL 2**

In [ ]:
# Switch model
cfg.EXPERIMENT_NAME = "fusion_model_2"
cfg.FUSION_STRATEGY = "model_2"

cfg.set_seed()

## **RUN K FOLD TRAINING (MODEL 2)**

In [ ]:
print("🚀 Running Model 2 Training...")
run_kfold(cfg)

## **RUN EXTERNAL VALIDATION (MODEL 2)**

In [ ]:
print("🌍 Running Model 2 External Validation...")
run_external(cfg)

## **LOAD RESULTS FOR COMPARISON**

In [ ]:
import numpy as np

# Model 1
m1_path = os.path.join(PROJECT_PATH, "outputs/fusion_model_1/predictions/external_predictions.npz")
m1 = np.load(m1_path)

# Model 2
m2_path = os.path.join(PROJECT_PATH, "outputs/fusion_model_2/predictions/external_predictions.npz")
m2 = np.load(m2_path)

y_true = m1["y_true"]

y_prob_m1 = m1["y_prob"]
y_pred_m1 = m1["y_pred"]

y_prob_m2 = m2["y_prob"]
y_pred_m2 = m2["y_pred"]

print("✅ Loaded predictions")

## **ROC OVERLAY (MODEL COMPARISON)**

In [ ]:
from src.evaluation.roc import ROCAnalysis

roc_path = os.path.join(PROJECT_PATH, "outputs/model_comparison_roc.png")

ROCAnalysis.plot_model_comparison(
    y_true,
    {
        "Model 1": y_prob_m1,
        "Model 2": y_prob_m2
    },
    roc_path
)

print("✅ ROC comparison saved")

## **INTERNAL vs EXTERNAL ROC (MODEL 1)**

In [ ]:
# Load internal predictions (Model 1 example)
internal_path = os.path.join(
    PROJECT_PATH,
    "outputs/fusion_model_1_final/predictions/aggregated_predictions.npz"
)

internal = np.load(internal_path)

y_true_int = internal["y_true"]
y_prob_int = internal["y_prob"]

roc_path = os.path.join(PROJECT_PATH, "outputs/internal_vs_external.png")

ROCAnalysis.plot_internal_vs_external_ci(
    y_true_int,
    y_prob_int,
    y_true,
    y_prob_m1,
    roc_path
)

print("✅ Internal vs External ROC saved")

## **DISPLAY FINAL TABLE**

In [ ]:
import pandas as pd

df = pd.read_csv(os.path.join(PROJECT_PATH, "outputs/model_comparison.csv"))
df

In [ ]:
from src.evaluation.statistics import Statistics

p_value = Statistics.delong_roc_test(
    y_true,
    y_prob_m1,
    y_prob_m2
)

print(f"\n📊 DeLong Test (Model 1 vs Model 2)")
print(f"p-value: {p_value:.6f}")
print(f"Significance: {Statistics.significance_label(p_value)}")